In [1]:
import numpy as np
import pandas as pd
from pulp import LpVariable

In [ ]:
# ---------------- 1. READ THE EXCEL SHEETS ----------------
# PC = procurement centre, Mill = rice mill, WH = state warehouse (SW), SP = state procurement storage, RH = rail head
import os
# resolve the data file relative to this notebook, so it works regardless of the current working directory
BASE_DIR = os.path.dirname(os.path.abspath(globals().get('__file__', os.getcwd())))
xl = pd.ExcelFile(os.path.join(BASE_DIR, 'Backend', 'Data_11.xlsx'))

PC = pd.read_excel(xl, 'A.1 Warehouse')
Mill = pd.read_excel(xl, 'A.2 FPS')
WH  = pd.read_excel(xl, 'A.2 Warehouse')
SP  = pd.read_excel(xl, 'A.3 Storage')
RH  = pd.read_excel(xl, 'A.3 RH')
RHMap = pd.read_excel(xl, 'RHMap')

In [3]:
# ---------------- 2. CLEAN DISTRICT NAMES ----------------
# lowercase + remove all spaces, so the same district is spelt the same way in every sheet
def clean(s):
    return s.astype(str).str.lower().str.replace(r'\s+', '', regex=True)

In [4]:
PC['District'] = clean(PC['WH_District'])
Mill['District'] = clean(Mill['FPS_District'])
WH['District']  = clean(WH['SW_District'])
SP['District']  = clean(SP['WH_District'])
RH['District']  = clean(RH['WH_District'])

In [5]:
# 'thiruvallore' (SP and RH sheets) is the same district as 'thiruvallur'
for df in (PC, Mill, WH, SP, RH):
    df['District'] = df['District'].replace({'thiruvallur': 'thiruvalluvar',
                                             'thiruvallore': 'thiruvalluvar'})

In [6]:
# ---------------- 3. SURPLUS / DEFICIT partition LABEL OF EVERY ROW ----------------
# the sheets already contain the label in these (unnamed) columns
PC['Region'] = PC['Unnamed: 8']
Mill['Region'] = Mill['Unnamed: 11']
WH['Region']  = WH['Unnamed: 8']
SP['Region']  = SP['Unnamed: 8']
RH['Region']  = RH['quantity of Wheat stored (Quintals)']

In [7]:
# every district gets ONE label (Surplus / Deficit), used for all node types
all_nodes = pd.concat([d[['District', 'Region']] for d in (PC, Mill, WH, SP, RH)])

# creating a dictionary mapping each district to its region (Surplus / Deficit)
district_region = (all_nodes.dropna().drop_duplicates().set_index('District')['Region'].to_dict())

# now we can assign the label to every row in every sheet, based on the district of that row
for df in (PC, Mill, WH, SP, RH):
    df['Region'] = df['District'].map(district_region)

surplus_districts = {d for d, r in district_region.items() if r == 'Surplus'}
deficit_districts = {d for d, r in district_region.items() if r == 'Deficit'}

# segregating the rows as idx of each table into surplus and deficit districts
# Example: PC_S = rows of the PC table in surplus districts, Mill_D = rows of Mill in deficit districts
def idx(df, r): 
    return list(df.index[df['Region'] == r])

PC_S, PC_D = idx(PC,'Surplus'), idx(PC,'Deficit')
Mill_S, Mill_D = idx(Mill,'Surplus'), idx(Mill,'Deficit')
SP_S,  SP_D  = idx(SP,'Surplus'),  idx(SP,'Deficit')
WH_S,  WH_D  = idx(WH,'Surplus'),  idx(WH,'Deficit')
RH_S,  RH_D  = idx(RH,'Surplus'),  idx(RH,'Deficit')

In [ ]:
# ---------------- 4. DISTANCE BETWEEN TWO POINTS (km, straight line on the earth) ----------------
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp, dl = p2 - p1, np.radians(lon2 - lon1)
    a = np.sin(dp/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

# haversine distance between srcdf(node 1) and dst_df(node 2)

In [ ]:
# ---------------- 5. PARAMETERS ----------------
d_km = 150    # FIXED threshold distance d:
              #   PC_S -> SP_D goes by ROAD  if road distance <= d_km
              #   PC_S -> SP_D goes by RAIL  if road distance >  d_km   (road is then not allowed: x_ij = 0)

a = 0.225      # mill rule: paddy coming DIRECTLY from PCs = a x mill capacity (surplus mills), <= a x capacity (deficit mills)
CONV = 0.67    # 1 quintal of paddy gives 0.67 quintal of rice

# ALLOW_EXTRA_AT_WH (REASON): True = a warehouse may receive MORE than its demand.
# The model is infeasible if this is False (tested), because every PC must be emptied.
ALLOW_EXTRA_AT_WH = True

# build_arc: distance of every allowed (source row i, destination row j) pair
#   max_km   : keep only pairs with distance < max_km
#   src_dcol / dst_dcol : SAME-DISTRICT RULE (professor's requirement 1).
#        When both are given, only pairs whose source district == destination district are kept,
#        so every mapping of PC/SP/Mill/RH stays inside one district (within its own surplus/deficit region).
#        FALLBACK: if a source row has NO destination in its own district (the data has no such node there),
#        its single NEAREST allowed destination is kept instead, so the model stays feasible.
#        These fallback arcs are the only cross-district arcs and are reported in the verification cell.
def build_arc(src_df, slat, slon, src_idx, dst_df, dlat, dlon, dst_idx,
              max_km=None, src_dcol=None, dst_dcol=None):
    src_idx, dst_idx = list(src_idx), list(dst_idx)
    # sl= source latitudes, so=source longitudes, dl=destination latitudes, do=destination longitudes
    sl, so = src_df.loc[src_idx, slat].to_numpy(), src_df.loc[src_idx, slon].to_numpy()
    dl, do = dst_df.loc[dst_idx, dlat].to_numpy(), dst_df.loc[dst_idx, dlon].to_numpy()
    D = haversine_km(sl[:, None], so[:, None], dl[None, :], do[None, :])   # matrix of all distances

    keep = np.ones(D.shape, dtype=bool)
    if max_km is not None:
        keep &= (D < max_km)

    # ---- SAME-DISTRICT rule + nearest-partner fallback ----
    if src_dcol is not None and dst_dcol is not None:
        sd = src_df.loc[src_idx, src_dcol].to_numpy().astype(str)
        dd = dst_df.loc[dst_idx, dst_dcol].to_numpy().astype(str)
        same = sd[:, None] == dd[None, :]
        keep_d = keep & same
        no_arc = ~keep_d.any(axis=1)                      # source rows with no same-district partner
        for r in np.nonzero(no_arc)[0]:
            allowed = np.nonzero(keep[r])[0]              # candidates that still respect max_km
            if allowed.size:
                keep_d[r, allowed[np.argmin(D[r, allowed])]] = True   # keep the nearest one
        keep = keep_d

    a_, b_ = np.nonzero(keep)
    return {(src_idx[x], dst_idx[y]): float(D[x, y]) for x, y in zip(a_, b_)}

In [ ]:
# ---------------- 6. ALL ROAD ARROWS OF THE FIGURE ----------------
# Distance[tag][(i, j)] = km from source row i to destination row j for a particular tag
# tag names: PCS = PC surplus, MILLS = mill surplus, SPD = SP deficit ... (S = surplus, D = deficit)
#
# SAME-DISTRICT RULE (professor's requirement 1): every WITHIN-REGION mapping is restricted to one district
# by passing the 'District' column as src_dcol / dst_dcol (DD below). The two CROSS-REGION bridges keep their
# original rule: PCS_SPD stays distance-based (<= d_km), and the surplus->deficit rail (RH_S -> RH_D) stays
# map-based (RHMap sheet) - neither can be same-district because the two ends are in different regions.

DD = dict(src_dcol='District', dst_dcol='District')   # turns the same-district rule on for a tag

Distance = {
 # ---- surplus region (same-district) ----
 'PCS_MILLS': build_arc(PC,'WH_Lat','WH_Long',PC_S,Mill,'FPS_Lat','FPS_Long',Mill_S, **DD),
 'PCS_SPS':   build_arc(PC,'WH_Lat','WH_Long',PC_S,SP,'WH_Lat','WH_Long',SP_S, **DD),
 'SPS_MILLS': build_arc(SP,'WH_Lat','WH_Long',SP_S,Mill,'FPS_Lat','FPS_Long',Mill_S, **DD),
 'MILLS_SWS': build_arc(Mill,'FPS_Lat','FPS_Long',Mill_S,WH,'SW_Lat','SW_Long',WH_S, **DD),
 'PCS_RHS':   build_arc(PC,'WH_Lat','WH_Long',PC_S,RH,'WH_Lat','WH_Long',RH_S, **DD),
 # ---- surplus -> deficit by ROAD: CROSS-REGION, distance rule only (pairs <= d_km) ----
 # (the far pairs, > d_km, go by rail and are added in the rail cell below)
 'PCS_SPD':   build_arc(PC,'WH_Lat','WH_Long',PC_S,SP,'WH_Lat','WH_Long',SP_D, max_km=d_km + 1e-6),
 # ---- deficit region (same-district) ----
 'PCD_SPD':   build_arc(PC,'WH_Lat','WH_Long',PC_D,SP,'WH_Lat','WH_Long',SP_D, **DD),
 'PCD_MILLD': build_arc(PC,'WH_Lat','WH_Long',PC_D,Mill,'FPS_Lat','FPS_Long',Mill_D, **DD),
 'SPD_MILLD': build_arc(SP,'WH_Lat','WH_Long',SP_D,Mill,'FPS_Lat','FPS_Long',Mill_D, **DD),
 'RHD_MILLD': build_arc(RH,'WH_Lat','WH_Long',RH_D,Mill,'FPS_Lat','FPS_Long',Mill_D, **DD),
 'MILLD_SWD': build_arc(Mill,'FPS_Lat','FPS_Long',Mill_D,WH,'SW_Lat','SW_Long',WH_D, **DD),
}

# NOT in this list on purpose:
#   RH_S -> RH_D (rail)   : Selected from RHMap sheet (no need to optimize)
#   RH_D -> SP_D          : rail paddy that must reach a storage is handled in the optimization model itself through RHMap sheet

In [ ]:
# ---------------- 6b. SUPPLY-AWARE FALLBACK for the surplus mill a-rule ----------------
# The surplus a-rule is an EQUALITY: direct PC->mill == a x capacity, and under the same-district rule
# that paddy can only come from PCs in the mill's own district. A few surplus districts do not have
# enough local PC supply to feed their mills (e.g. thiruchirappalli, thiruvalluvar, nagapattinam,
# pudukkottai). For those districts we connect the mills to the NEAREST out-of-district surplus PCs,
# just enough to cover the shortfall (with a margin). These are the only cross-district PCS_MILLS arcs
# and are listed in the verification cell. (Surplus PC emptying itself is never blocked: every surplus
# PC can always export to the deficit region by rail via PCS_SPD_RAIL.)
def add_supply_fallback(tagd, src_df, slat, slon, src_idx, dst_df, dlat, dlon, dst_idx,
                        supply_col, cap_col, a, margin=1.3):
    src_idx, dst_idx = list(src_idx), list(dst_idx)
    src_by_d = defaultdict(list)
    for i in src_idx: src_by_d[src_df.at[i, 'District']].append(i)
    dst_by_d = defaultdict(list)
    for j in dst_idx: dst_by_d[dst_df.at[j, 'District']].append(j)
    # FREE PC supply per district = local supply beyond that district's OWN mill a-need (committed first).
    # The short districts are geographically clustered, so we must draw from genuinely free supply (big
    # surplus districts) and track it GLOBALLY so two short districts never claim the same free supply.
    free_left = {}
    for d, pcs in src_by_d.items():
        sup  = sum(src_df.at[i, supply_col] for i in pcs)
        need = a * sum(dst_df.at[j, cap_col] for j in dst_by_d.get(d, []))
        free_left[d] = max(0.0, sup - need)
    shorts = []
    for dist, mills in dst_by_d.items():
        local = sum(src_df.at[i, supply_col] for i in src_by_d.get(dist, []))
        short = a * sum(dst_df.at[j, cap_col] for j in mills) - local
        if short > 1: shorts.append((short, dist, mills))
    shorts.sort(reverse=True)                          # largest shortfall first
    added = 0
    for short, dist, mills in shorts:
        mlat = dst_df.loc[mills, dlat].to_numpy(); mlon = dst_df.loc[mills, dlon].to_numpy()
        cand = []
        for i in src_idx:
            d = src_df.at[i, 'District']
            if d == dist or free_left.get(d, 0) <= 0:
                continue
            dmin = haversine_km(src_df.at[i, slat], src_df.at[i, slon], mlat, mlon).min()
            cand.append((dmin, i))
        cand.sort()                                   # nearest out-of-district PCs that still have free supply
        acc = 0.0
        for dmin, i in cand:
            d = src_df.at[i, 'District']
            if free_left.get(d, 0) <= 0:
                continue
            for j in mills:                           # connect this PC to every mill in the short district
                if (i, j) not in tagd:
                    tagd[(i, j)] = float(haversine_km(src_df.at[i, slat], src_df.at[i, slon],
                                                      dst_df.at[j, dlat], dst_df.at[j, dlon]))
                    added += 1
            take = min(src_df.at[i, supply_col], free_left[d])
            free_left[d] -= take; acc += take
            if acc >= short * margin:
                break
    return added

# ---- ABSORPTION fallback (deficit side): connect each deficit PC / SP to its k NEAREST deficit mills ----
# Deficit PCs/SPs cannot export (no rail out of the deficit region), so every quintal they hold must reach
# a deficit mill. Some deficit districts have NO mill (theni, ramanathapuram) or too little capacity
# (thoothukudi), which strands paddy under the strict same-district rule. Connecting each deficit PC/SP to
# its k nearest deficit mills guarantees enough reachable milling capacity. The NEAREST mill is in the
# node's own district whenever one exists, so the optimum still keeps the flow inside the district; the
# extra arcs only carry flow where the local district cannot absorb it (reported in the verification cell).
def connect_knn(tagd, src_df, slat, slon, src_idx, dst_df, dlat, dlon, dst_idx, k=6):
    dla = dst_df.loc[dst_idx, dlat].to_numpy(); dlo = dst_df.loc[dst_idx, dlon].to_numpy()
    dst_arr = list(dst_idx); added = 0
    for i in src_idx:
        Di = haversine_km(src_df.at[i, slat], src_df.at[i, slon], dla, dlo)
        for pos in np.argsort(Di)[:k]:
            j = dst_arr[pos]
            if (i, j) not in tagd:
                tagd[(i, j)] = float(Di[pos]); added += 1
    return added

from collections import defaultdict
added = add_supply_fallback(Distance['PCS_MILLS'], PC,'WH_Lat','WH_Long',PC_S,
                            Mill,'FPS_Lat','FPS_Long',Mill_S, 'Paddy_Procurement','Mill_Capacity', a)
a1 = connect_knn(Distance['PCD_MILLD'], PC,'WH_Lat','WH_Long',PC_D, Mill,'FPS_Lat','FPS_Long',Mill_D, k=6)
a2 = connect_knn(Distance['SPD_MILLD'], SP,'WH_Lat','WH_Long',SP_D, Mill,'FPS_Lat','FPS_Long',Mill_D, k=6)
print("supply-aware fallback: added", added, "PCS_MILLS arcs (surplus a-rule)")
print("absorption fallback:   added", a1, "PCD_MILLD +", a2, "SPD_MILLD arcs (deficit overflow)")

In [ ]:
# ---------------- 7. RAIL HEAD MAP (sheet RHMap) ----------------
# columns used: From_RH (surplus rail head), To_RH (deficit rail head), Qty = paddy moved on that pair
RHMap = RHMap.rename(columns={RHMap.columns[2]: 'From_RH', RHMap.columns[5]: 'To_RH', RHMap.columns[8]: 'Qty'})
RHMap['From_RH'] = RHMap['From_RH'].astype(str)
RHMap['To_RH'] = RHMap['To_RH'].astype(str)
RH['WH_ID'] = RH['WH_ID'].astype(str)

# below code gives row_number, name of station from A3.RH sheet which is used to map the From_RH and To_RH columns to the particular station they belong to.
# dictionary flips the map as 'ID' -> row number
rh_pos = {rid: i for i, rid in RH['WH_ID'].items()}   
RHMap['i'] = RHMap['From_RH'].map(rh_pos)
RHMap['j'] = RHMap['To_RH'].map(rh_pos)

unmatched = RHMap[RHMap['i'].isna() | RHMap['j'].isna()]
print(f"RHMap rows not usable (rail head ID not in A.3 RH): {len(unmatched)}")   # VNRRH001 / VNRRH002 are missing

RHMap rows not usable (rail head ID not in A.3 RH): 10


In [30]:
# ---------------- 8. RAIL: FIXED SPLIT OF EVERY SURPLUS RAIL HEAD ----------------
# RULE: whatever reaches surplus rail head r leaves it to EVERY deficit rail head s listed for r in RHMap,
#       in the proportions of the sheet's quantity column:
#       share[r,s] = Qty(r->s) / sum of Qty of all pairs leaving r
# The model does NOT choose this split and rail has no cost (a rail head simply passes the paddy on).
# Pairs whose rail head is missing in the RH sheet are dropped, shares are computed among the remaining pairs.
from collections import defaultdict

usable = RHMap.dropna(subset=['i', 'j', 'Qty']).copy()
usable['i'] = usable['i'].astype(int)
usable['j'] = usable['j'].astype(int)
usable = usable[usable['i'].isin(RH_S) & usable['j'].isin(RH_D)]       # surplus RH -> deficit RH only
usable['share'] = usable['Qty'] / usable.groupby('i')['Qty'].transform('sum')

dest_of = defaultdict(list)    # r -> list of (s, share)   : where surplus rail head r sends its paddy
src_of  = defaultdict(list)    # s -> list of (r, share)   : who sends paddy to deficit rail head s
for row in usable.itertuples():
    dest_of[row.i].append((row.j, row.share))
    src_of[row.j].append((row.i, row.share))
print("usable rail pairs:", len(usable), "| surplus rail heads with a map:", len(dest_of))

usable rail pairs: 54 | surplus rail heads with a map: 14


In [31]:
# ---------------- 9. FAR PAIRS PC_S -> SP_D GO BY RAIL ----------------
# RULE for every pair (PC_S i, SP_D j):
#   road distance <= d_km : ROAD only  (already in Distance['PCS_SPD'], cost = road distance)
#   road distance >  d_km : RAIL only  (road variable does not exist = x_ij = 0), cost =
#        dist(PC i -> surplus rail head r)  +  sum over s of share[r,s] * dist(deficit rail head s -> SP j)
#        (rail between the two rail heads is free; the paddy reaches SP j from every s in its fixed shares)
# Which surplus rail head r? the one with the smallest cost (chosen here in advance, the model does not choose).
pc_lat, pc_lon = PC.loc[PC_S,'WH_Lat'].to_numpy(), PC.loc[PC_S,'WH_Long'].to_numpy()
sp_lat, sp_lon = SP.loc[SP_D,'WH_Lat'].to_numpy(), SP.loc[SP_D,'WH_Long'].to_numpy()
rh_lat, rh_lon = RH['WH_Lat'].to_numpy(), RH['WH_Long'].to_numpy()

Droad = haversine_km(pc_lat[:,None], pc_lon[:,None], sp_lat[None,:], sp_lon[None,:])   # all PC_S x SP_D road distances
near = Droad < d_km + 1e-6                                                                # True = road pair

rail_r = sorted(dest_of)                          # surplus rail heads that have a mapping in the sheet
costs = []
for r in rail_r:
    leg1 = haversine_km(pc_lat, pc_lon, rh_lat[r], rh_lon[r])                                              # PC i -> RH r
    leg2 = sum(sh * haversine_km(rh_lat[s], rh_lon[s], sp_lat, sp_lon) for s, sh in dest_of[r])         # RH s -> SP j
    costs.append(leg1[:, None] + leg2[None, :])
costs = np.stack(costs)
best_r, best_cost = costs.argmin(axis=0), costs.min(axis=0)

Distance['PCS_SPD_RAIL'] = {}      # cost (km) of the far pairs
RailVia = {}                       # (i, j) -> surplus rail head used by this pair
for a_, b_ in zip(*np.nonzero(~near)):
    i, j = PC_S[a_], SP_D[b_]
    Distance['PCS_SPD_RAIL'][(i, j)] = float(best_cost[a_, b_])
    RailVia[(i, j)] = rail_r[best_r[a_, b_]]

In [32]:
# ---------------- 10. DECISION VARIABLES ----------------
# Allocation[tag][(i, j)] = quantity (quintals of paddy, or of rice for the Mill -> WH arrows) sent from i to j
Allocation = {
    tag: {(i, j): LpVariable(f"X_{tag}_{i}_{j}", lowBound=0, cat='Continuous') for (i, j) in dd}
    for tag, dd in Distance.items()
}

print("surplus districts:", len(surplus_districts), "| deficit districts:", len(deficit_districts))
for tag, aa in Allocation.items():
    print(f"{tag:14s}{len(aa):>10,}")
print("TOTAL variables:", f"{sum(len(aa) for aa in Allocation.values()):,}")

surplus districts: 13 | deficit districts: 24
PCS_MILLS         51,739
PCS_SPS           39,680
SPS_MILLS          5,256
MILLS_SWS          5,238
PCS_RHS           27,776
PCS_SPD           35,887
PCD_SPD           10,480
PCD_MILLD         14,493
SPD_MILLD          7,280
RHD_MILLD          4,732
MILLD_SWD          7,550
PCS_SPD_RAIL      97,041
TOTAL variables: 307,152


In [33]:
# ======================================================================
#                    MODEL: constraints and objective
# ======================================================================
from collections import defaultdict
from pulp import LpProblem, LpMinimize, lpSum, LpStatus, value, CPLEX_CMD

model = LpProblem("Wheat_Distribution_Model", LpMinimize)

BIG = 10000    # penalty per quintal of unmet demand (much bigger than any distance, so it is used only if unavoidable)
Mill_S_set = set(Mill_S)

# helper: collects the variables of a node in ONE pass (much faster than looping over all variables for every node)
#   spec = list of (tag, end);  end = 0 -> node is the SOURCE of the arrow,  end = 1 -> node is the DESTINATION
#   returns {node row number: [variables]}
def flows(spec):
    dct = defaultdict(list)
    for tag, end in spec:
        for key, var in Allocation[tag].items():
            dct[key[end]].append(var)
    return dct

# ===============================================================
# 0. UNMET DEMAND (slack variable) - WHY WE NEED IT
# ===============================================================
# According to the Excel data the demand cannot be fully met:
#     total paddy procured = 17.82 M quintals  -> x 0.67 = about 11.94 M rice
#     total WH annual demand               = about 12.10 M rice
# so the state is about 0.16 M short, whatever the model does.
# If we forced "rice delivered == demand" the model would be INFEASIBLE and give no answer at all.
# So every WH gets a variable Unmet[j] >= 0 with:   rice received + Unmet = demand.
# Unmet is punished heavily in the objective (BIG), so the model meets as much demand as it can,
# and Unmet shows exactly where, and how much, the data cannot satisfy.
Unmet = {j: LpVariable(f"Unmet_{j}", lowBound=0) for j in WH_S + WH_D}

# ===============================================================
# OBJECTIVE: minimise quantity x distance (ROAD only) + unmet penalty
# ===============================================================
# Every arrow in Allocation is a road arrow with its distance, including:
#   PCS_SPD       : direct road PC_S -> SP_D (pairs <= d_km), cost = road distance
#   PCS_SPD_RAIL  : far pairs by rail, cost = the two road legs (PC -> RH_S and RH_D -> SP)
# Rail between the two rail heads is NOT in the objective: a rail head simply passes on whatever it receives
# to its mapped rail heads (fixed shares from the sheet), so there is nothing to optimise there.
objective = lpSum(Distance[t][k] * Allocation[t][k] for t in Allocation for k in Allocation[t]) \
            + BIG * lpSum(Unmet.values())
model += objective, "Total_Transport_Distance"

# helper lists for the far (rail) pairs
XR = Allocation['PCS_SPD_RAIL']
XR_by_i = defaultdict(list)      # far-pair variables of each PC
XR_by_j = defaultdict(list)      # far-pair variables arriving at each SP
XR_via_r = defaultdict(list)     # far-pair variables routed through each surplus rail head
for (i, j), v in XR.items():
    XR_by_i[i].append(v); XR_by_j[j].append(v); XR_via_r[RailVia[(i, j)]].append(v)

# ===============================================================
# 1. PC (procurement centres): everything procured must leave
# ===============================================================
# Both surplus and deficit PCs are emptied: total entering = total leaving,
# i.e. everything sent out (to mills, SPs, rail heads, road / rail to deficit SPs) == Paddy_Procurement.
pc_out = flows([('PCS_MILLS',0),('PCS_SPS',0),('PCS_RHS',0),('PCS_SPD',0),
                ('PCD_SPD',0),('PCD_MILLD',0)])
for i in set(pc_out) | set(XR_by_i):
    model += lpSum(pc_out[i]) + lpSum(XR_by_i[i]) == PC.at[i, 'Paddy_Procurement'], f"PC_supply_{i}"

# ===============================================================
# 2. MILLS: a-rule, capacity, rice balance
# ===============================================================
mill_pc_in = flows([('PCS_MILLS',1),('PCD_MILLD',1)])                       # paddy coming DIRECTLY from PCs
mill_in    = flows([('PCS_MILLS',1),('SPS_MILLS',1),('PCD_MILLD',1),('SPD_MILLD',1),('RHD_MILLD',1)])   # all paddy in
mill_out   = flows([('MILLS_SWS',0),('MILLD_SWD',0)])                       # rice going to WH
for m in set(mill_in) | set(mill_out):
    cap = Mill.at[m, 'Mill_Capacity']
    if mill_pc_in[m]:
        if m in Mill_S_set:      # surplus mill: paddy directly from PCs == a x capacity
            model += lpSum(mill_pc_in[m]) == a * cap, f"Mill_a_rule_{m}"
        else:                    # deficit mill: paddy directly from PCs <= a x capacity
            model += lpSum(mill_pc_in[m]) <= a * cap, f"Mill_a_rule_{m}"
    if mill_in[m]:               # total paddy in (PC + SP + RH) cannot exceed the mill capacity
        model += lpSum(mill_in[m]) <= cap, f"Mill_cap_{m}"
    if mill_in[m] or mill_out[m]:  # rice sent to WH = 0.67 x paddy received
        model += lpSum(mill_out[m]) == CONV * lpSum(mill_in[m]), f"Mill_balance_{m}"

# ===============================================================
# 3. SP (state procurement storage): in = out, capacity
# ===============================================================
# Deficit SPs also receive the paddy of the far PC_S -> SP_D pairs (XR_by_j).
sp_in  = flows([('PCS_SPS',1),('PCS_SPD',1),('PCD_SPD',1)])
sp_out = flows([('SPS_MILLS',0),('SPD_MILLD',0)])
for s_ in set(sp_in) | set(sp_out) | set(XR_by_j):
    inflow = lpSum(sp_in[s_]) + lpSum(XR_by_j[s_])
    if sp_out[s_] or len(inflow) > 0:
        model += inflow == lpSum(sp_out[s_]), f"SP_balance_{s_}"      # a storage keeps nothing: in = out
    if len(inflow) > 0:
        model += inflow <= SP.at[s_, 'Capacity'], f"SP_cap_{s_}"

# ===============================================================
# 4. RAIL HEADS: fixed split from the sheet, balance, capacity
# ===============================================================
# Surplus rail head r receives
#    (a) paddy from PCs that goes on to deficit MILLS         -> Allocation['PCS_RHS']
#    (b) paddy of far PC_S -> SP_D pairs routed through r      -> XR_via_r
# Everything that enters r leaves to the deficit rail heads s in the FIXED shares of the sheet,
# so deficit rail head s receives  share[r,s] x (inflow of r)  from every r that is mapped to it.
# Balance (a rail head stores nothing): at deficit rail head s, part (b) continues to the SPs
# (already counted in the SP balance above), and part (a) must leave to deficit mills (RHD_MILLD).
# Capacity (300,000 per rail head): a rail head physically handles only this much paddy. It matters here:
#   13 surplus rail heads x 300,000 = 3.9 M, while the deficit region needs about 3.3 to 3.6 M of paddy
#   from the surplus region, so the capacity can really bind. Without it, all paddy could pass through one rail head.
pcrh_in = flows([('PCS_RHS',1)])
rhd_out = flows([('RHD_MILLD',0)])

in_r = {}                                     # total paddy entering each surplus rail head
for r in RH_S:
    in_r[r] = lpSum(pcrh_in[r]) + lpSum(XR_via_r[r])
    if len(in_r[r]) == 0:
        continue
    if r in dest_of:
        model += in_r[r] <= RH.at[r, 'Capacity'], f"RH_cap_{r}"
    else:   # surplus rail head with no mapping in the sheet cannot forward anything (nothing may enter it)
        model += in_r[r] == 0, f"RH_nomap_{r}"

for s_ in RH_D:
    inflow_total = lpSum(sh * in_r[r] for r, sh in src_of[s_])                  # all paddy reaching s
    inflow_mill  = lpSum(sh * lpSum(pcrh_in[r]) for r, sh in src_of[s_])         # part (a): goes on to mills
    if len(inflow_mill) > 0 or rhd_out[s_]:
        model += inflow_mill == lpSum(rhd_out[s_]), f"RH_balance_{s_}"
    if len(inflow_total) > 0:
        model += inflow_total <= RH.at[s_, 'Capacity'], f"RH_cap_{s_}"

# ===============================================================
# 5. WH (state warehouses): demand, with unmet slack (see section 0)
# ===============================================================
# rice received + Unmet >= demand      (ALLOW_EXTRA_AT_WH = True)
# WHY ">=" and not "==" (tested on your data): every PC must be emptied, and every surplus mill must take exactly
# a x capacity directly from PCs. Together this forces more rice into some warehouses than their annual demand,
# and with "received <= demand" the model is INFEASIBLE. So a warehouse may hold extra rice;
# the extra is not penalised and the model never sends extra on purpose (it costs distance).
# Unmet[j] is then the shortfall of warehouse j (see the results cell for shortfall and extra).
wh_in = flows([('MILLS_SWS',1),('MILLD_SWD',1)])
for j in WH_S + WH_D:
    if ALLOW_EXTRA_AT_WH:
        model += lpSum(wh_in[j]) + Unmet[j] >= WH.at[j, 'Annual_Demand'], f"WH_demand_{j}"
    else:
        model += lpSum(wh_in[j]) + Unmet[j] == WH.at[j, 'Annual_Demand'], f"WH_demand_{j}"

print("Variables:", len(model.variables()), "| Constraints:", len(model.constraints))

Variables: 307404 | Constraints: 4973


In [34]:
# ======================================================================
#                              SOLVE
# ======================================================================
cplex_path = r"C:\Program Files\IBM\ILOG\CPLEX_Studio222\cplex\bin\x64_win64\cplex.exe"

model.writeLP("model.lp")

# --- Solve ---
# The model is a pure LP (no integer variables), so the 'mipgap' option is not needed.
status = model.solve(CPLEX_CMD(path=cplex_path, msg=True, timeLimit=1800, logPath="cplex_log.txt"))   # log is also saved in cplex_log.txt
print("Status:", LpStatus[model.status])

Status: Optimal


In [36]:
# ======================================================================
#                              RESULTS
# ======================================================================
# After solving the model
print("Objective Value =", value(model.objective), "(= transport + penalty, see below)")

# the objective also contains BIG x unmet, so the real transport cost is shown separately
transport = sum(Distance[t][k] * (Allocation[t][k].varValue or 0) for t in Allocation for k in Allocation[t])
print("Transport only (quintal-km) =", round(transport))
print("Penalty part (BIG x unmet)  =", round(BIG * sum(v.varValue or 0 for v in Unmet.values())))
print("Total unmet demand =", sum(v.varValue or 0 for v in Unmet.values()))
print("PC_S -> SP_D by road:", sum(v.varValue or 0 for v in Allocation['PCS_SPD'].values()))
print("PC_S -> SP_D by rail:", sum(v.varValue or 0 for v in Allocation['PCS_SPD_RAIL'].values()))

# total flow and quintal-km of every arrow
for tag, dd in Allocation.items():
    tot = sum(v.varValue or 0 for v in dd.values())
    km  = sum((v.varValue or 0) * Distance[tag][k] for k, v in dd.items())
    print(f"{tag:14s} flow = {tot:>14,.0f}   quintal-km = {km:>16,.0f}")

# extra rice received above demand (only possible when ALLOW_EXTRA_AT_WH = True)
wh_recv = {j: sum((v.varValue or 0) for k, v in Allocation['MILLS_SWS'].items() if k[1] == j) +
              sum((v.varValue or 0) for k, v in Allocation['MILLD_SWD'].items() if k[1] == j) for j in WH_S + WH_D}
extra = sum(max(0, wh_recv[j] - WH.at[j, 'Annual_Demand']) for j in wh_recv)
print("Total rice received above demand (extra):", round(extra))
print("Total rice received:", round(sum(wh_recv.values())), "| total annual demand:", round(WH['Annual_Demand'].sum()))

# rice received vs demand, region by region
for name, ids in (('Surplus', WH_S), ('Deficit', WH_D)):
    rec = sum(wh_recv[j] for j in ids)
    dem = WH.loc[ids, 'Annual_Demand'].sum()
    print(f"{name}: rice received {rec:,.0f} vs demand {dem:,.0f}  ->  {rec - dem:+,.0f}")

# warehouses that could not be fully served
short = [(WH.at[j, 'SW_Name'], round(v.varValue)) for j, v in Unmet.items() if (v.varValue or 0) > 1]
print("WH with unmet demand:", len(short))
print(sorted(short, key=lambda x: -x[1])[:15])

Objective Value = 2286892188.7546015 (= transport + penalty, see below)
Transport only (quintal-km) = 657199910
Penalty part (BIG x unmet)  = 1629692279
Total unmet demand = 162969.22790158112
PC_S -> SP_D by road: 813954.578820777
PC_S -> SP_D by rail: 115170.4
PCS_MILLS      flow =      7,468,661   quintal-km =      171,573,412
PCS_SPS        flow =      3,565,875   quintal-km =       43,036,110
SPS_MILLS      flow =      3,565,875   quintal-km =       29,165,128
MILLS_SWS      flow =      7,393,139   quintal-km =      123,005,298
PCS_RHS        flow =      2,401,214   quintal-km =       34,398,299
PCS_SPD        flow =        813,955   quintal-km =       26,085,473
PCD_SPD        flow =      1,353,838   quintal-km =       20,433,142
PCD_MILLD      flow =      2,102,162   quintal-km =       52,712,014
SPD_MILLD      flow =      2,282,963   quintal-km =       45,149,941
RHD_MILLD      flow =      2,401,214   quintal-km =       35,289,078
MILLD_SWD      flow =      4,546,847   quintal-

In [37]:
from collections import defaultdict
import pandas as pd

load_r = defaultdict(float)                       # paddy entering every SURPLUS rail head
for (i, r), v in Allocation['PCS_RHS'].items():
    load_r[r] += v.varValue or 0
for (i, j), v in Allocation['PCS_SPD_RAIL'].items():
    load_r[RailVia[(i, j)]] += v.varValue or 0
load_s = defaultdict(float)                       # paddy reaching every DEFICIT rail head (fixed shares of the sheet)
for r, l in load_r.items():
    for s, sh in dest_of.get(r, []):
        load_s[s] += sh * l

rows = []
for h in RH_S + RH_D:
    load = load_r[h] if h in RH_S else load_s[h]
    rows.append({'RH_ID': RH.at[h, 'WH_ID'], 'District': RH.at[h, 'District'], 'Region': RH.at[h, 'Region'],
                 'Load': round(load), 'Capacity': RH.at[h, 'Capacity'], 'Util_%': round(100 * load / RH.at[h, 'Capacity'])})
rh_table = pd.DataFrame(rows).sort_values('Util_%', ascending=False)
print(rh_table.to_string(index=False))

  RH_ID       District  Region   Load  Capacity  Util_%
    C02      thanjavur Surplus 300000    300000     100
    C03      thanjavur Surplus 300000    300000     100
    B80 mayiladuthurai Surplus 300000    300000     100
KRIRH01    krishnagiri Deficit 300000    300000     100
 SLMRH1          salem Deficit 300000    300000     100
    B66     dharmapuri Deficit 300000    300000     100
    B91       tiruppur Deficit 300000    300000     100
CHNRH01       chennain Deficit 300000    300000     100
RHCUD01      cuddalore Surplus 296948    300000      99
    B81 mayiladuthurai Surplus 293135    300000      98
    B56          erode Deficit 263526    300000      88
    C30       namakkal Deficit 247949    300000      83
    957       ariyalur Surplus 225203    300000      75
CPTRH01   chengalpattu Surplus 220149    300000      73
TVRRH04     thiruvarur Surplus 199057    300000      66
    C38        ranipet Surplus 185061    300000      62
TNVRH01   thirunelveli Deficit 171967    300000 

In [ ]:
# ======================================================================
#       CONSTRAINT VERIFICATION (PASS / FAIL) + NODE-TYPE TOTALS
#       (professor's requirement 2: show totals per node type and prove every constraint holds)
# ======================================================================
from collections import defaultdict
TOL = 1.0                      # quintals: tolerance for LP floating-point rounding
def Vv(v): return v.varValue or 0.0

# which tags carry the same-district rule, and (src_df, dst_df) to read districts from
RESTRICTED = {'PCS_MILLS':(PC,Mill), 'PCS_SPS':(PC,SP), 'SPS_MILLS':(SP,Mill), 'MILLS_SWS':(Mill,WH),
              'PCS_RHS':(PC,RH), 'PCD_SPD':(PC,SP), 'PCD_MILLD':(PC,Mill), 'SPD_MILLD':(SP,Mill),
              'RHD_MILLD':(RH,Mill), 'MILLD_SWD':(Mill,WH)}
# fallback arcs = the only allowed cross-district arcs (where the data had no same-district partner)
fb_keys = {t: {(i,j) for (i,j) in Allocation[t]
               if sdf.at[i,'District'] != ddf.at[j,'District']} for t,(sdf,ddf) in RESTRICTED.items()}

# ---- per-node flow sums (one pass) ----
out_pc=defaultdict(float); in_mill=defaultdict(float); pcin_mill=defaultdict(float)
out_mill=defaultdict(float); in_sp=defaultdict(float); out_sp=defaultdict(float); in_wh=defaultdict(float)
for t in ('PCS_MILLS','PCS_SPS','PCS_RHS','PCS_SPD','PCS_SPD_RAIL','PCD_SPD','PCD_MILLD'):
    for (i,j),v in Allocation[t].items(): out_pc[i]+=Vv(v)
for t in ('PCS_MILLS','SPS_MILLS','PCD_MILLD','SPD_MILLD','RHD_MILLD'):
    for (i,j),v in Allocation[t].items(): in_mill[j]+=Vv(v)
for t in ('PCS_MILLS','PCD_MILLD'):
    for (i,j),v in Allocation[t].items(): pcin_mill[j]+=Vv(v)
for t in ('MILLS_SWS','MILLD_SWD'):
    for (i,j),v in Allocation[t].items(): out_mill[i]+=Vv(v); in_wh[j]+=Vv(v)
for t in ('PCS_SPS','PCS_SPD','PCD_SPD','PCS_SPD_RAIL'):
    for (i,j),v in Allocation[t].items(): in_sp[j]+=Vv(v)
for t in ('SPS_MILLS','SPD_MILLD'):
    for (i,j),v in Allocation[t].items(): out_sp[i]+=Vv(v)
# rail head realized loads (fixed shares of the sheet)
load_r=defaultdict(float)
for (i,r),v in Allocation['PCS_RHS'].items(): load_r[r]+=Vv(v)
for (i,j),v in Allocation['PCS_SPD_RAIL'].items(): load_r[RailVia[(i,j)]]+=Vv(v)
load_s=defaultdict(float)
for r,l in load_r.items():
    for s,sh in dest_of.get(r,[]): load_s[s]+=sh*l
pcrh_r=defaultdict(float)
for (i,r),v in Allocation['PCS_RHS'].items(): pcrh_r[r]+=Vv(v)
rhd_out_v=defaultdict(float)
for (r,m),v in Allocation['RHD_MILLD'].items(): rhd_out_v[r]+=Vv(v)
mill_part_s=defaultdict(float)
for s in RH_D:
    for r,sh in src_of[s]: mill_part_s[s]+=sh*pcrh_r[r]

results=[]
def chk(name, ok, detail=""): results.append((name, "PASS" if ok else "FAIL", detail))
mills_pcarc={j for t in ('PCS_MILLS','PCD_MILLD') for (i,j) in Allocation[t]}

bad=[i for i in PC_S+PC_D if abs(out_pc[i]-PC.at[i,'Paddy_Procurement'])>TOL]
chk("1. Every PC emptied (out == procurement)", not bad, f"{len(bad)} off")
bad=[m for m in Mill_S if m in mills_pcarc and abs(pcin_mill[m]-a*Mill.at[m,'Mill_Capacity'])>TOL]
chk("2a. Surplus mill direct-from-PC == a*cap", not bad, f"{len(bad)} off")
bad=[m for m in Mill_D if m in mills_pcarc and pcin_mill[m]-a*Mill.at[m,'Mill_Capacity']>TOL]
chk("2b. Deficit mill direct-from-PC <= a*cap", not bad, f"{len(bad)} over")
bad=[m for m in in_mill if in_mill[m]-Mill.at[m,'Mill_Capacity']>TOL]
chk("3. Mill total paddy-in <= capacity", not bad, f"{len(bad)} over")
bad=[m for m in set(in_mill)|set(out_mill) if abs(out_mill[m]-CONV*in_mill[m])>TOL]
chk("4. Mill rice-out == 0.67 * paddy-in", not bad, f"{len(bad)} off")
bad=[s for s in set(in_sp)|set(out_sp) if abs(in_sp[s]-out_sp[s])>TOL]
chk("5a. SP inflow == outflow (stores nothing)", not bad, f"{len(bad)} off")
bad=[s for s in in_sp if in_sp[s]-SP.at[s,'Capacity']>TOL]
chk("5b. SP inflow <= capacity", not bad, f"{len(bad)} over")
bad=[r for r in RH_S if load_r[r]-RH.at[r,'Capacity']>TOL]
chk("6a. Surplus rail-head load <= capacity", not bad, f"{len(bad)} over")
bad=[s for s in RH_D if load_s[s]-RH.at[s,'Capacity']>TOL]
chk("6b. Deficit rail-head load <= capacity", not bad, f"{len(bad)} over")
bad=[s for s in RH_D if abs(mill_part_s[s]-rhd_out_v[s])>TOL]
chk("6c. Deficit rail-head balance (to-mill part == out)", not bad, f"{len(bad)} off")
bad=[j for j in WH_S+WH_D if in_wh[j]+Vv(Unmet[j])-WH.at[j,'Annual_Demand']< -TOL]
chk("7. WH rice + unmet >= demand", not bad, f"{len(bad)} short")
bad=[k for k in Allocation['PCS_SPD'] if Distance['PCS_SPD'][k]>d_km+1e-3]
chk("8. Road PC_S->SP_D only if dist <= 150 km", not bad, f"{len(bad)} over")
viol=sum(1 for t,(sdf,ddf) in RESTRICTED.items() for (i,j),v in Allocation[t].items()
         if sdf.at[i,'District']!=ddf.at[j,'District'] and Vv(v)>TOL and (i,j) not in fb_keys[t])
chk("9. Cross-district flow only on fallback arcs", viol==0, f"{viol} unexpected")

print("================ CONSTRAINT CHECKS ================")
for n,s,d in results:
    print(f"  [{s}] {n}" + (f"   ({d})" if s=='FAIL' and d else ""))
print("  >>> ALL CONSTRAINTS SATISFIED <<<" if all(s=='PASS' for _,s,_ in results) else "  >>> SOME CHECKS FAILED <<<")

print("\n================ NODE-TYPE TOTALS (quintals) ================")
print(f"PC   total out          : {sum(out_pc.values()):>14,.0f}   (supply {PC['Paddy_Procurement'].sum():,.0f})")
print(f"SP   in / out           : {sum(in_sp.values()):>14,.0f} / {sum(out_sp.values()):,.0f}   (gap {sum(in_sp.values())-sum(out_sp.values()):,.0f})")
print(f"Mill paddy-in / rice-out: {sum(in_mill.values()):>14,.0f} / {sum(out_mill.values()):,.0f}   (ratio {sum(out_mill.values())/max(sum(in_mill.values()),1):.3f} vs 0.670)")
print(f"RH   surplus / deficit  : {sum(load_r.values()):>14,.0f} / {sum(load_s.values()):,.0f}")
print(f"WH   rice-in            : {sum(in_wh.values()):>14,.0f}   (demand {WH['Annual_Demand'].sum():,.0f}, unmet {sum(Vv(v) for v in Unmet.values()):,.0f})")

print("\n================ SAME-DISTRICT RULE: fallback (cross-district) arcs ================")
print("(these are the ONLY cross-district arcs, used because the data has no same-district partner)")
for t,(sdf,ddf) in RESTRICTED.items():
    if fb_keys[t]:
        flow=sum(Vv(v) for (i,j),v in Allocation[t].items() if (i,j) in fb_keys[t])
        dists=sorted({sdf.at[i,'District'] for (i,j) in fb_keys[t]})
        print(f"  {t:12s}: {len(fb_keys[t]):>3} arcs, flow = {flow:>12,.1f}   districts: {', '.join(dists)}")